# 09 · Audit AI-Face v2

Notebook này kiểm tra dataset AI-Face trước khi tạo subset pilot và chạy unseen-generator benchmark.

**Phạm vi:** chỉ audit annotation, generator split, class balance, archive và image mẫu. Notebook không train model và không tạo subset cuối cùng.

## Quyết định protocol

Pilot hiện tại dùng 8 generator:

- GAN: `taming_transformer_VQGAN`, `stylegan3`, `AttGAN`, `STARGAN`;
- Diffusion: `StableDiffusion1.5`, `latent_diffusion`, `Palette`, `StableDiffusion_Inpainting`;
- Real pilot: `imdb_wiki`;
- Không dùng `deepfakes/` trong main experiment.

Mỗi generator sẽ lấy tối đa 2.000 ảnh fake. Split generator-disjoint sẽ được tạo ở bước subset builder sau khi audit pass.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import json
import os
import re
import shutil
import subprocess
import zipfile

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from PIL import Image
except ImportError:
    Image = None

# Local default. Override this cell on Kaggle if the dataset is mounted elsewhere.
DATA_ROOT = Path("/Users/nguyentran0703/Downloads/FaceTrace/data/AI_Face")
if not DATA_ROOT.exists():
    DATA_ROOT = Path("../data/AI_Face")

ANNOT_ROOT = DATA_ROOT / "AI_Face_annotationsV2"
TRAIN_CSV = ANNOT_ROOT / "train_data.csv"
TEST_CSV = ANNOT_ROOT / "test_data.csv"
OUT_ROOT = DATA_ROOT / "audit_outputs"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SELECTED = {
    ("GANs", "taming_transformer_VQGAN"): "train",
    ("GANs", "stylegan3"): "train",
    ("GANs", "AttGAN"): "val",
    ("GANs", "STARGAN"): "test",
    ("DMs", "StableDiffusion1.5"): "train",
    ("DMs", "latent_diffusion"): "train",
    ("DMs", "Palette"): "val",
    ("DMs", "StableDiffusion_Inpainting"): "test",
}

print("DATA_ROOT:", DATA_ROOT)
print("Annotation exists:", TRAIN_CSV.exists(), TEST_CSV.exists())
print("PIL available:", Image is not None)


DATA_ROOT: /Users/nguyentran0703/Downloads/FaceTrace/data/AI_Face
Annotation exists: True True
PIL available: True


## 1. Đọc và chuẩn hóa annotation

`train_data.csv` và `test_data.csv` của AI-Face là image-level split. Ta gộp chúng để kiểm tra generator overlap thay vì coi `test_data.csv` là unseen test.

In [2]:
def normalize_row(row, source_split):
    path = (row.get("Image Path") or "").replace("\\", "/")
    parts = [part for part in path.split("/") if part]
    family = next((name for name in ("GANs", "DMs", "deepfakes", "Real") if name in parts), "<unknown>")
    idx = parts.index(family) if family in parts else -1
    generator = parts[idx + 1] if idx >= 0 and idx + 1 < len(parts) else "<unknown>"
    if family == "DMs" and generator == "CommercialTools" and idx + 2 < len(parts):
        generator = f"CommercialTools/{parts[idx + 2]}"

    basename = Path(path).name
    identity_match = re.match(r"(nm\d+|\d+)_", basename)
    identity_hint = identity_match.group(1) if identity_match else None

    return {
        "path": path,
        "source_split": source_split,
        "family": family,
        "generator": generator,
        "family_generator": (family, generator),
        "target": row.get("Target"),
        "identity_hint": identity_hint,
        "row": row,
    }


def read_annotation(path, source_split):
    records = []
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        fields = reader.fieldnames
        for row in reader:
            records.append(normalize_row(row, source_split))
    return fields, records

train_fields, train_records = read_annotation(TRAIN_CSV, "train_data")
test_fields, test_records = read_annotation(TEST_CSV, "test_data")
records = train_records + test_records

print("Train rows:", len(train_records))
print("Test rows:", len(test_records))
print("All rows:", len(records))
print("Train fields:", train_fields)
print("Test fields:", test_fields)


Train rows: 1317236
Test rows: 329309
All rows: 1646545
Train fields: ['Image Path', 'Predicted Gender', 'Predicted Age', 'Skin Tone', 'Target', 'Skin Tone Group', 'Intersection']
Test fields: ['Image Path', 'Predicted Gender', 'Predicted Age', 'Reliability Score Gender', 'Reliability Score Age', 'Skin Tone', 'Target', 'Skin Tone Group', 'Intersection']


In [3]:
def counts_by(records, key):
    return Counter(item[key] for item in records)

print("Target counts:", counts_by(records, "target"))
print("Family counts:", counts_by(records, "family"))
print("Exact path duplicates in combined annotation:", len(records) - len({item["path"] for item in records}))

all_paths = {item["path"] for item in train_records}
test_paths = {item["path"] for item in test_records}
print("Exact train/test path overlap:", len(all_paths & test_paths))

if pd is not None:
    summary_rows = []
    for (family, generator), role in sorted(SELECTED.items()):
        subset = [item for item in records if item["family_generator"] == (family, generator)]
        summary_rows.append({
            "family": family,
            "generator": generator,
            "pilot_role": role,
            "total": len(subset),
            "fake": sum(item["target"] == "1" for item in subset),
            "real": sum(item["target"] == "0" for item in subset),
            "default_train": sum(item["source_split"] == "train_data" for item in subset),
            "default_test": sum(item["source_split"] == "test_data" for item in subset),
        })
    generator_table = pd.DataFrame(summary_rows)
    display(generator_table)
    generator_table.to_csv(OUT_ROOT / "generator_counts.csv", index=False)


Target counts: Counter({'1': 1245660, '0': 400885})


Family counts: Counter({'DMs': 595890, 'deepfakes': 421920, 'GANs': 319780, 'Real': 308955})


Exact path duplicates in combined annotation: 0
Exact train/test path overlap: 0


,family,generator,pilot_role,total,fake,real,default_train,default_test
0,DMs,Palette,val,6000,6000,0,4804,1196
1,DMs,StableDiffusion1.5,train,18111,18111,0,14428,3683
2,DMs,StableDiffusion_Inpainting,test,20989,20989,0,16749,4240
3,DMs,latent_diffusion,train,20000,20000,0,15917,4083
4,GANs,AttGAN,val,6005,6005,0,4823,1182
5,GANs,STARGAN,test,5648,5648,0,4482,1166
6,GANs,stylegan3,train,26770,26770,0,21416,5354
7,GANs,taming_transformer_VQGAN,train,50000,50000,0,39943,10057


## 2. Kiểm tra generator overlap của split mặc định

Nếu generator xuất hiện ở cả hai CSV thì split mặc định không thể dùng cho unseen-generator evaluation.

In [4]:
train_generators = {item["family_generator"] for item in train_records}
test_generators = {item["family_generator"] for item in test_records}
overlap = train_generators & test_generators
print("Train generator groups:", len(train_generators))
print("Test generator groups:", len(test_generators))
print("Overlapping groups:", len(overlap))
print(sorted(overlap))

split_rows = []
for key in sorted(train_generators | test_generators):
    family, generator = key
    n_train = sum(item["family_generator"] == key and item["source_split"] == "train_data" for item in records)
    n_test = sum(item["family_generator"] == key and item["source_split"] == "test_data" for item in records)
    split_rows.append({
        "family": family,
        "generator": generator,
        "train_rows": n_train,
        "test_rows": n_test,
        "train_test_overlap": n_train > 0 and n_test > 0,
    })

if pd is not None:
    split_table = pd.DataFrame(split_rows)
    display(split_table)
    split_table.to_csv(OUT_ROOT / "default_split_generator_overlap.csv", index=False)


Train generator groups: 24
Test generator groups: 24
Overlapping groups: 24
[('DMs', 'CommercialTools/dalle2'), ('DMs', 'CommercialTools/if'), ('DMs', 'CommercialTools/midjourney'), ('DMs', 'DCFACe'), ('DMs', 'Palette'), ('DMs', 'StableDiffusion1.5'), ('DMs', 'StableDiffusion_Inpainting'), ('DMs', 'latent_diffusion'), ('GANs', 'AttGAN'), ('GANs', 'MMD_GAN'), ('GANs', 'MSG_STYLE_GAN'), ('GANs', 'ProGAN images'), ('GANs', 'STARGAN'), ('GANs', 'STGAN'), ('GANs', 'STYLEGAN'), ('GANs', 'StyleGAN2'), ('GANs', 'stylegan3'), ('GANs', 'taming_transformer_VQGAN'), ('Real', 'FFHQ'), ('Real', 'imdb_wiki'), ('deepfakes', 'celebdf'), ('deepfakes', 'dfd'), ('deepfakes', 'dfdc'), ('deepfakes', 'ff++')]


,family,generator,train_rows,test_rows,train_test_overlap
0,DMs,CommercialTools/dalle2,158,46,True
1,DMs,CommercialTools/if,396,109,True
2,DMs,CommercialTools/midjourney,86,14,True
3,DMs,DCFACe,424452,105529,True
4,DMs,Palette,4804,1196,True
5,DMs,StableDiffusion1.5,14428,3683,True
6,DMs,StableDiffusion_Inpainting,16749,4240,True
7,DMs,latent_diffusion,15917,4083,True
8,GANs,AttGAN,4823,1182,True
9,GANs,MMD_GAN,795,205,True


## 3. Kiểm tra identity heuristic của `imdb_wiki`

Annotation không có identity ID chính thức. Cell này chỉ tạo một `identity_hint` từ filename để phát hiện rủi ro; không được coi là bằng chứng identity-disjoint.

In [5]:
real_records = [item for item in records if item["family_generator"] == ("Real", "imdb_wiki")]
identity_by_split = {}
for source_split in ("train_data", "test_data"):
    identity_by_split[source_split] = {
        item["identity_hint"]
        for item in real_records
        if item["source_split"] == source_split and item["identity_hint"]
    }

print("Real imdb rows:", len(real_records))
print("Rows with filename identity hint:", sum(item["identity_hint"] is not None for item in real_records))
print("Train identity hints:", len(identity_by_split["train_data"]))
print("Test identity hints:", len(identity_by_split["test_data"]))
print("Heuristic overlap:", len(identity_by_split["train_data"] & identity_by_split["test_data"]))
print("Interpretation: filename hints are diagnostic only; identity-disjointness is not proven.")


Real imdb rows: 238955
Rows with filename identity hint: 238955
Train identity hints: 33043
Test identity hints: 12765
Heuristic overlap: 6342
Interpretation: filename hints are diagnostic only; identity-disjointness is not proven.


## 4. Kiểm tra archive

Cell này kiểm tra các inner archive cần cho pilot có tồn tại trong hai outer archive hay không. Full CRC test có thể mất vài phút; bật `RUN_FULL_CRC_TEST` nếu muốn chạy lại.

In [6]:
OUTER_ARCHIVES = {
    DATA_ROOT / "OneDrive_1_10-7-2026.zip": {
        "taming_transformer_VQGAN.zip",
        "stylegan3.zip",
        "AttGAN.zip",
        "STARGAN.zip",
    },
    DATA_ROOT / "OneDrive_2_10-7-2026.zip": {
        "StableDiffusion1.5.zip",
        "latent_diffusion.zip",
        "Palette.zip",
        "StableDiffusion_Inpainting.zip",
    },
}

archive_rows = []
for outer, selected_members in OUTER_ARCHIVES.items():
    if not outer.exists():
        print("Missing outer archive:", outer)
        continue
    with zipfile.ZipFile(outer) as archive:
        names = set(archive.namelist())
    for member in sorted(selected_members):
        archive_rows.append({
            "outer_archive": outer.name,
            "inner_archive": member,
            "present": member in names,
        })
    print(outer.name, "selected present:", sorted(selected_members & names), "missing:", sorted(selected_members - names))

real_archive = DATA_ROOT / "imdb_wiki.zip"
print("imdb_wiki.zip exists:", real_archive.exists())
if real_archive.exists():
    with zipfile.ZipFile(real_archive) as archive:
        image_members = [name for name in archive.namelist() if name.lower().endswith((".jpg", ".jpeg", ".png"))]
    print("imdb_wiki image members:", len(image_members))

if pd is not None:
    archive_table = pd.DataFrame(archive_rows)
    display(archive_table)
    archive_table.to_csv(OUT_ROOT / "archive_members.csv", index=False)


OneDrive_1_10-7-2026.zip selected present: ['AttGAN.zip', 'STARGAN.zip', 'stylegan3.zip', 'taming_transformer_VQGAN.zip'] missing: []
OneDrive_2_10-7-2026.zip selected present: ['Palette.zip', 'StableDiffusion1.5.zip', 'StableDiffusion_Inpainting.zip', 'latent_diffusion.zip'] missing: []
imdb_wiki.zip exists: True


imdb_wiki image members: 238955


,outer_archive,inner_archive,present
0,OneDrive_1_10-7-2026.zip,AttGAN.zip,True
1,OneDrive_1_10-7-2026.zip,STARGAN.zip,True
2,OneDrive_1_10-7-2026.zip,stylegan3.zip,True
3,OneDrive_1_10-7-2026.zip,taming_transformer_VQGAN.zip,True
4,OneDrive_2_10-7-2026.zip,Palette.zip,True
5,OneDrive_2_10-7-2026.zip,StableDiffusion1.5.zip,True
6,OneDrive_2_10-7-2026.zip,StableDiffusion_Inpainting.zip,True
7,OneDrive_2_10-7-2026.zip,latent_diffusion.zip,True


In [7]:
RUN_FULL_CRC_TEST = False
if RUN_FULL_CRC_TEST:
    for archive_path in [*OUTER_ARCHIVES.keys(), real_archive]:
        if not archive_path.exists():
            continue
        with zipfile.ZipFile(archive_path) as archive:
            bad_member = archive.testzip()
        print(archive_path.name, "CRC:", "PASS" if bad_member is None else f"FAIL: {bad_member}")
else:
    print("CRC test skipped. Set RUN_FULL_CRC_TEST=True to run it.")


CRC test skipped. Set RUN_FULL_CRC_TEST=True to run it.


## 5. Decode ảnh mẫu

Cell này dùng `unzip` + `bsdtar` nếu có sẵn để stream-extract một ảnh từ mỗi inner archive, không giải nén toàn bộ dataset. Kích thước gốc được ghi lại vì các generator có thể dùng resolution khác nhau.

In [8]:
def command_exists(name):
    return shutil.which(name) is not None

CAN_STREAM_NESTED = command_exists("unzip") and command_exists("bsdtar")
SAMPLE_DIR = OUT_ROOT / "sample_images"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
print("Can stream nested archives:", CAN_STREAM_NESTED)


def first_member_from_nested(outer, inner):
    command = f"unzip -p {shlex_quote(str(outer))} {shlex_quote(inner)} | bsdtar -tf - | rg -m1 '\.(jpg|jpeg|png)$'"
    result = subprocess.run(command, shell=True, capture_output=True, text=True, check=True)
    return result.stdout.strip()


def extract_member_from_nested(outer, inner, member, output_path):
    command = f"unzip -p {shlex_quote(str(outer))} {shlex_quote(inner)} | bsdtar -xOf - {shlex_quote(member)}"
    with output_path.open("wb") as handle:
        subprocess.run(command, shell=True, stdout=handle, check=True)


def shlex_quote(value):
    import shlex
    return shlex.quote(value)


sample_rows = []
if CAN_STREAM_NESTED:
    for outer, selected_members in OUTER_ARCHIVES.items():
        for inner in sorted(selected_members):
            try:
                member = first_member_from_nested(outer, inner)
                output_path = SAMPLE_DIR / (inner.replace(".zip", "") + ".sample")
                extract_member_from_nested(outer, inner, member, output_path)
                row = {"archive": inner, "member": member, "path": str(output_path), "status": "extracted"}
                if Image is not None:
                    with Image.open(output_path) as image:
                        image.load()
                        row.update({"format": image.format, "width": image.width, "height": image.height, "mode": image.mode})
                sample_rows.append(row)
            except Exception as exc:
                sample_rows.append({"archive": inner, "status": f"ERROR: {exc}"})
else:
    print("Skip nested sample extraction: unzip/bsdtar unavailable.")

if real_archive.exists() and command_exists("bsdtar"):
    try:
        result = subprocess.run(
            f"bsdtar -tf {shlex_quote(str(real_archive))} | rg -m1 '\.(jpg|jpeg|png)$'",
            shell=True, capture_output=True, text=True, check=True,
        )
        member = result.stdout.strip()
        output_path = SAMPLE_DIR / "imdb_wiki.sample"
        with output_path.open("wb") as handle:
            subprocess.run(f"bsdtar -xOf {shlex_quote(str(real_archive))} {shlex_quote(member)}", shell=True, stdout=handle, check=True)
        row = {"archive": "imdb_wiki.zip", "member": member, "path": str(output_path), "status": "extracted"}
        if Image is not None:
            with Image.open(output_path) as image:
                image.load()
                row.update({"format": image.format, "width": image.width, "height": image.height, "mode": image.mode})
        sample_rows.append(row)
    except Exception as exc:
        sample_rows.append({"archive": "imdb_wiki.zip", "status": f"ERROR: {exc}"})

if pd is not None:
    sample_table = pd.DataFrame(sample_rows)
    display(sample_table)
    sample_table.to_csv(OUT_ROOT / "sample_decode.csv", index=False)


Can stream nested archives: True


<>:11: SyntaxWarning: invalid escape sequence '\.'
<>:49: SyntaxWarning: invalid escape sequence '\.'
<>:11: SyntaxWarning: invalid escape sequence '\.'
<>:49: SyntaxWarning: invalid escape sequence '\.'
/var/folders/nx/tm25jbz54jjdb9mb8c28s0m00000gn/T/ipykernel_60854/476313921.py:11: SyntaxWarning: invalid escape sequence '\.'
  command = f"unzip -p {shlex_quote(str(outer))} {shlex_quote(inner)} | bsdtar -tf - | rg -m1 '\.(jpg|jpeg|png)$'"
/var/folders/nx/tm25jbz54jjdb9mb8c28s0m00000gn/T/ipykernel_60854/476313921.py:49: SyntaxWarning: invalid escape sequence '\.'
  f"bsdtar -tf {shlex_quote(str(real_archive))} | rg -m1 '\.(jpg|jpeg|png)$'",


,archive,member,path,status,format,width,height,mode
0,AttGAN.zip,AttGAN/8364.png,/Users/nguyentran0703/Downloads/FaceTrace/data...,extracted,PNG,256,256,RGB
1,STARGAN.zip,STARGAN/image_5110.png,/Users/nguyentran0703/Downloads/FaceTrace/data...,extracted,PNG,256,256,RGB
2,stylegan3.zip,stylegan3_T_FFHQU_processed/seed41572_face1.png,/Users/nguyentran0703/Downloads/FaceTrace/data...,extracted,PNG,244,244,RGB
3,taming_transformer_VQGAN.zip,taming_transformer:VQGAN/tt-ffhq/ffhq_k300_p1....,/Users/nguyentran0703/Downloads/FaceTrace/data...,extracted,JPEG,200,200,RGB
4,Palette.zip,Palette_CelebAHQ/images/img003463.jpg,/Users/nguyentran0703/Downloads/FaceTrace/data...,extracted,JPEG,200,200,RGB
5,StableDiffusion1.5.zip,StableDiffusion1.5_wiki_processed_process/08/7...,/Users/nguyentran0703/Downloads/FaceTrace/data...,extracted,PNG,244,244,RGB
6,StableDiffusion_Inpainting.zip,StableDiffusion_Inpainting_wiki_processed_proc...,/Users/nguyentran0703/Downloads/FaceTrace/data...,extracted,PNG,244,244,RGB
7,latent_diffusion.zip,latent_diffusion_FFHQ/guided-diffusion_noise2i...,/Users/nguyentran0703/Downloads/FaceTrace/data...,extracted,PNG,256,256,RGB
8,imdb_wiki.zip,imdb_wiki/imdb/43/nm0000843_rm4063200512_1931-...,/Users/nguyentran0703/Downloads/FaceTrace/data...,extracted,JPEG,244,244,RGB


## 6. Quality gates và quyết định

Các gate này chỉ đánh giá dataset/protocol, chưa đánh giá model.

In [9]:
selected_counts = {
    key: sum(item["family_generator"] == key for item in records)
    for key in SELECTED
}

checks = {
    "all_selected_generators_present": all(count >= 2000 for count in selected_counts.values()),
    "default_split_is_not_generator_disjoint": len(overlap) > 0,
    "combined_exact_path_duplicates_zero": len(records) == len({item["path"] for item in records}),
    "selected_target_is_fake": all(
        all(item["target"] == "1" for item in records if item["family_generator"] == key)
        for key in SELECTED
    ),
    "selected_archive_members_present": all(row["present"] for row in archive_rows),
}

print(json.dumps(checks, indent=2, ensure_ascii=False))
with (OUT_ROOT / "audit_summary.json").open("w", encoding="utf-8") as handle:
    json.dump({
        "data_root": str(DATA_ROOT),
        "train_rows": len(train_records),
        "test_rows": len(test_records),
        "selected_counts": {f"{family}/{generator}": count for (family, generator), count in selected_counts.items()},
        "checks": checks,
        "identity_disjointness_verified": False,
        "notes": [
            "Default train/test split is image-level and generator-overlapping.",
            "Pilot uses a custom generator-disjoint split.",
            "Real pilot uses imdb_wiki only.",
        ],
    }, handle, indent=2, ensure_ascii=False)

assert checks["all_selected_generators_present"]
assert checks["combined_exact_path_duplicates_zero"]
assert checks["selected_target_is_fake"]
assert checks["selected_archive_members_present"]
print("Dataset audit metadata/archive gates: PASS")
print("Remaining gate: create final subset manifest and run full duplicate/readability audit.")


{
  "all_selected_generators_present": true,
  "default_split_is_not_generator_disjoint": true,
  "combined_exact_path_duplicates_zero": true,
  "selected_target_is_fake": true,
  "selected_archive_members_present": true
}
Dataset audit metadata/archive gates: PASS
Remaining gate: create final subset manifest and run full duplicate/readability audit.


## 7. Final subset quality gate

Sau khi subset được tạo và duplicate được sửa, quality gate cuối cùng được đọc từ artifact sinh bởi script.

In [10]:
SUBSET_ROOT = DATA_ROOT / "aiface_pilot_32k"
QUALITY_PATH = SUBSET_ROOT / "audit" / "quality_gate.json"
if QUALITY_PATH.exists():
    subset_quality = json.loads(QUALITY_PATH.read_text(encoding="utf-8"))
    print(json.dumps(subset_quality, indent=2, ensure_ascii=False))
    assert subset_quality["passed"]
    print("Final subset quality gate: PASS")
else:
    print("Final subset quality gate has not been run yet:", QUALITY_PATH)


{
  "n_rows": 32000,
  "n_unique_sha256": 32000,
  "n_duplicate_groups": 0,
  "split_counts": {
    "test": 8000,
    "train": 16000,
    "val": 8000
  },
  "label_counts": {
    "test/0": 4000,
    "test/1": 4000,
    "train/0": 8000,
    "train/1": 8000,
    "val/0": 4000,
    "val/1": 4000
  },
  "generator_splits": {
    "AttGAN": [
      "val"
    ],
    "Palette": [
      "val"
    ],
    "STARGAN": [
      "test"
    ],
    "StableDiffusion1.5": [
      "train"
    ],
    "StableDiffusion_Inpainting": [
      "test"
    ],
    "latent_diffusion": [
      "train"
    ],
    "stylegan3": [
      "train"
    ],
    "taming_transformer_VQGAN": [
      "train"
    ]
  },
  "real_identity_counts": {
    "test": 2599,
    "train": 4218,
    "val": 2669
  },
  "real_identity_overlap": {
    "train/val": 1383,
    "train/test": 1384,
    "val/test": 1027
  },
  "mode_counts": {
    "RGB": 32000
  },
  "issues": [],
  "passed": true
}
Final subset quality gate: PASS
